# Exercise 1

In [ ]:
def read_file(file_path: Path) -> str:
  return file_path.read_text()

critique_agent = Agent(MODEL, output_type=str, deps_type=Path, tools=[read_file])

@critique_agent.system_prompt
def system_prompt(ctx: RunContext[Path]) -> str:
  return f"Given a script at this path, critique the correctness and structure of it :{ctx.deps}"

result = critique_agent.run_sync(deps=coding_result.output)
pprint(result.output)

# Exercise 1 - alternative

In [ ]:
class ScriptInfo(BaseModel):
    script_content: str
    script_spec: ScriptSpec

class CodeCritique(BaseModel):
    critique: str = Field(description="The critique of the code")
    high_quality: bool = Field(description="Whether the code is high-quality")

critique_agent = Agent(
    MODEL,
    deps_type=ScriptInfo,
    output_type=CodeCritique,
)
@critique_agent.system_prompt
def system_prompt(ctx: RunContext[ScriptInfo]) -> str:
    return (
        f"Given this script, critique the correctness and structure of it."
        f"Also judge whether it's high quality of not. The script must adhere to the spec."
        f"Script content: {ctx.deps.script_content}"
        f"Original script spec: {ctx.deps.script_spec}"
    )

In [ ]:
spec_result = spec_maker.run_sync(
    user_prompt=
        "Create a clustering script (1 cluster per cell type) based on the gene experssion, then make a visualization (plots only) " \
        "called clusters_analysis.png that will show if our cell type annotation makes good clusters.",
    deps=Path('data/sc_expression')
)

code_is_high_quality = False
coding_result = None
critique=None
while not code_is_high_quality:
    print('Coding agent starting')
    coding_result = coding_agent.run_sync(
        deps=spec_result.output,
        retries=1,
        user_prompt=
            (
                f"Adjust the code based on this critique: {critique}"
                f"your previous code: {coding_result.output.read_text()}" 
            ) if critique is not None and coding_result is not None else None
    )
    critique_result = critique_agent.run_sync(
        deps=ScriptInfo(script_content=coding_result.output.read_text(), script_spec=spec_result.output)
    )
    code_is_high_quality = critique_result.output.high_quality
    critique = critique_result.output.critique

if coding_result is not None:
    print(coding_result.output)